In [87]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import io
import joblib
from pathlib import Path
from minio.error import S3Error
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from minio import Minio
from statsmodels.stats.proportion import proportions_ztest

## Configure dependencies access

In [88]:
BASE_PATH = Path.cwd().parent.parent
ENV_PATH = BASE_PATH / "hospital_production" / ".env"

load_dotenv(ENV_PATH)

# PostgreSQL connection parameters
POSTGRES_USER: str = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD: str = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB: str = os.getenv("POSTGRES_DB")
POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: str = os.getenv("POSTGRES_PORT")
POSTGRES_SCHEMA: str = os.getenv("POSTGRES_SCHEMA")
POSTGRES_TABLE: str = os.getenv("POSTGRES_TABLE")

# Build connection string
DATABASE_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

# Create SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# Open connection and execute query
with engine.connect() as conn:
    query = text(f"SELECT * FROM {POSTGRES_SCHEMA}.{POSTGRES_TABLE}")

    result = conn.execute(query)

    # Fetch and view the results into a pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

## Postgresql data readable

In [89]:
pd.set_option('display.max_columns', None)
df.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn,capacity_pressure,admission_rate_row,refusal_rate_row,staff_attendance_rate
0,PAT-f9c8afa6,Victor Baker,50,2025-07-30,2025-08-13,general_medicine,91,7,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,14,1,2.512195,0.388350,0.611650,0.821429
1,PAT-5290be70,Jeffrey Chandler,29,2025-11-01,2025-11-14,emergency,88,11,1,32,76.0,32.0,44.0,67.0,70.0,39,35.0,4.0,13,1,2.303030,0.421053,0.578947,0.897436
2,PAT-003ce690,Larry Dixon,29,2025-01-19,2025-01-21,ICU,60,1,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,2,1,1.000000,0.952381,0.047619,0.000000
3,PAT-18f78014,Kenneth Scott,77,2025-06-24,2025-06-26,general_medicine,61,6,4,43,152.0,43.0,109.0,67.0,66.0,28,22.0,6.0,2,1,3.454545,0.282895,0.717105,0.785714
4,PAT-69dc0dc1,April Frost,37,2025-11-15,2025-11-19,general_medicine,60,11,3,37,58.0,37.0,21.0,95.0,63.0,28,0.0,28.0,4,1,1.526316,0.637931,0.362069,0.000000


## Load ML Models from MinIO Bucket

In [90]:
BASE_PATH = Path.cwd().parent.parent
ENV_PATH = BASE_PATH / "hospital_production" / ".env"
load_dotenv(ENV_PATH)

True

In [91]:
# Intialize MinIO client
client = Minio(
    "localhost:9000",
    access_key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    secure=False  # Set to True if using HTTPS
)

# Define bucket
bucket_name = "hospital-models"

try:
    objects = client.list_objects(bucket_name, recursive=True)

    print(f"Model found in bucket '{bucket_name}':")
    for obj in objects:
        print(f" - {obj.object_name}")

except S3Error as e:
    print(f"Error occurred: {e}")

Model found in bucket 'hospital-models':
 - manifest/integrity_audit.json
 - manifest/patient_churn.json
 - manifest/risk_level.json
 - manifest/risk_score.json
 - manifest/service_performance.json
 - manifest/staff_staff_attendance_rate.json
 - manifest/staff_staff_morale.json
 - mlflow/1/137ee340884442a3a4709a03addbd778/artifacts/manifest_patient_churn.json
 - mlflow/1/86df9eacc59448e9bf19e80f594256fc/artifacts/manifest_risk_score.json
 - mlflow/1/9dce29aeaa1b4ca989b5eebe3e3ea306/artifacts/manifest_risk_level.json
 - mlflow/1/a583e441f1854ba7bc7999648306dc02/artifacts/manifest_staff_staff_attendance_rate.json
 - mlflow/1/ad552a5926c94c16a52d6ee0bfc2b8bc/artifacts/manifest_staff_staff_morale.json
 - mlflow/1/d8aa349d747c405a8010ff4844039fa9/artifacts/manifest_service_performance.json
 - mlflow/1/eb8b7cc09b85488b830e232387e35045/artifacts/manifest_integrity_audit.json
 - mlflow/1/models/m-02dbdf35d0c345c9a5ec60af384ec796/artifacts/MLmodel
 - mlflow/1/models/m-02dbdf35d0c345c9a5ec60af38

## Implemented models:
* ### Use PostgreSQL database for fetching the result
* ### Return to A/B Testing
* ### Implementation models to database and return the result

## A/B Testing

In [92]:
df["service"].value_counts()

service
emergency           2694
surgery             2557
general_medicine    2402
ICU                 2347
Name: count, dtype: int64

In [93]:
# Select comparison groups by respective model names
SERVICE_A = "emergency"
SERVICE_B = "surgery"
SERVICE_C = "general_medicine"
SERVICE_D = "ICU"

# Preprating data for z-test
data = df[df["service"].isin([SERVICE_A, SERVICE_B, SERVICE_C, SERVICE_D])].copy()

# Check whether counts are valid
if (data["patients_request"] < 0).any():
    raise ValueError("Negative patients request found.")

if (data["patients_admitted"] < 0).any():
    raise ValueError("Negative patients admitted found.")

# Aggregate counts data by service
summary = (
    data.groupby("service")
    .agg(
        visitors=("patients_request", "sum"),
        conversions=("patients_admitted", "sum")
    )
)

# Split into A and B groups
A = summary.loc[SERVICE_A]
B = summary.loc[SERVICE_B]
C = summary.loc[SERVICE_C]
D = summary.loc[SERVICE_D]

requests_A, admitted_A = int(A["visitors"]), int(A["conversions"])
requests_B, admitted_B = int(B["visitors"]), int(B["conversions"])
requests_C, admitted_C = int(C["visitors"]), int(C["conversions"])
requests_D, admitted_D = int(D["visitors"]), int(D["conversions"])

### Calculate admission rates + Two-proprotion Z-test

In [94]:
# Calculate admission rates 
rate_A = admitted_A / requests_A
rate_B = admitted_B / requests_B
rate_C = admitted_C / requests_C
rate_D = admitted_D / requests_D

absolute_difference = rate_B - rate_A
relative_lift = absolute_difference / rate_A if rate_A > 0 else np.nan

# Two proportion z-test
counts = np.asarray([admitted_D, admitted_C])
sample_sizes = np.asarray([requests_D, requests_C])

z_stat, p_value = proportions_ztest(counts, sample_sizes, alternative='two-sided')

# Approximate 95% confidence interval for rate difference (B - A)
se = np.sqrt(rate_A * (1 - rate_A) / requests_A) + (rate_B * (1 - rate_B) / requests_B)

ci_lower = absolute_difference - 1.96 * se
ci_upper = absolute_difference + 1.96 * se

print(f"Admission Rate for {SERVICE_A}: {rate_A:.4f}")
print(f"Admission Rate for {SERVICE_B}: {rate_B:.4f}")
print(f"Admission Rate for {SERVICE_C}: {rate_C:.4f}")
print(f"Admission Rate for {SERVICE_D}: {rate_D:.4f}")
print("=" * 55)
print(f"Z-statistic: {z_stat:.4f}, P-value: {p_value:.4f}")
print(f"Confidence interval for the difference in admission rates (B - A) -> Confidence lower: ({ci_lower:.4f}, Confidence upper: {ci_upper:.4f})")

Admission Rate for emergency: 0.1818
Admission Rate for surgery: 0.5507
Admission Rate for general_medicine: 0.2811
Admission Rate for ICU: 0.8772
Z-statistic: 244.8987, P-value: 0.0000
Confidence interval for the difference in admission rates (B - A) -> Confidence lower: (0.3677, Confidence upper: 0.3700)


### Table results

In [95]:
results_df = pd.DataFrame({
    "Metric": [
        "Patient Requests",
        "Patients Admitted",
        "Admission Rate",
        "Absolute Difference",
        "Relative Lift",
        "Z Statistic",
        "P-Value",
        "95% CI (D - C)"
    ],
    SERVICE_A: [
        f"{requests_A:,}",
        f"{admitted_A:,}",
        f"{rate_A:.2%}",
        f"{absolute_difference:+.2%}",
        f"{relative_lift:+.2%}" if np.isfinite(relative_lift) else "N/A",
        f"{z_stat:.4f}",
        f"{p_value:.4f}",
        f"[{ci_lower:.2%}, {ci_upper:.2%}]"
    ],
    SERVICE_B: [
        f"{requests_D:,}",
        f"{admitted_D:,}",
        f"{rate_D:.2%}",
        f"-",
        f"-",
        f"-",
        f"-",
        f"-"
    ]
}).set_index("Metric")

print("-" * 50)
print("Statistical Inference Results:")
print("-" * 50)
display(results_df)

--------------------------------------------------
Statistical Inference Results:
--------------------------------------------------


,emergency,surgery
Metric,,
Patient Requests,"449,635","43,103"
Patients Admitted,"81,763","37,809"
Admission Rate,18.18%,87.72%
Absolute Difference,+36.88%,-
Relative Lift,+202.84%,-
Z Statistic,244.8987,-
P-Value,0.0000,-
95% CI (D - C),"[36.77%, 37.00%]",-


### Statistical interpretation

In [96]:
# Assume alpha = 0.05
ALPHA = 0.05

print("\n" + "=" * 55)
print("HOSPITAL ADMISSION RATE ANALYSIS")
print("=" * 55)

if p_value < ALPHA:
    print("Statistically significant difference detected")

    if rate_B > rate_A:
        print(f"{SERVICE_B} has the higher observed admission rate")
    else:
        print(f"{SERVICE_A} has the higher observed admission rate")
else:
    print("No statistically significant difference detected")
    print("The observed difference may be consistent xwith sampling variability or random chance.")

print(f"\nP-value: {p_value:.5f}")
print(f"Admission rate difference: {absolute_difference:+.2%}")
print(f"95% CI: [{ci_lower:.2%}, {ci_upper:.2%}]")
print(
    "\nIMPORTANT: This is a comparison between services, "
    "not evidence of a causal treatment effect."
)


HOSPITAL ADMISSION RATE ANALYSIS
Statistically significant difference detected
surgery has the higher observed admission rate

P-value: 0.00000
Admission rate difference: +36.88%
95% CI: [36.77%, 37.00%]

IMPORTANT: This is a comparison between services, not evidence of a causal treatment effect.


## =====================================================

## Feature engineering

In [97]:
# Build required column helper if there missing dependency
df["risk_score"] = (
        (df["patients_request"] <= df["patients_request"].mean()).astype(float)
        + (df["patients_refused"] >= df["patients_refused"].mean()).astype(float)
        + (df["patients_admitted"] <= df["patients_admitted"].mean()).astype(float)
        + (df["satisfaction"] <= df["satisfaction"].mean()).astype(float)
    ) / 4.0
df["risk_level"] = pd.cut(df['risk_score'], bins=[-1, 0.25, 0.5, 0.75, 1], labels=['Low', 'Medium', 'High', 'Critical'])
df["service_performance"] = (df["patient_satisfaction"] * df["patients_admitted"] - df["patients_refused"]) / df["patients_request"]
df["staff_absent"] = df["staff_absens"]
df["health_score"] = (
    0.40 * (df["satisfaction"] / 100)
    + 0.20 * (1 - df["patient_churn"])
    + 0.20 * df["admission_rate_row"]
    + 0.20 * df["staff_attendance_rate"]
) * 100

# Avoid inf in service_performance by replacing inf with NaN and then filling NaN with 0
df["service_performance"].replace([np.inf, -np.inf], np.nan, inplace=True)
df['admission_rate_row'].replace([np.inf, -np.inf], np.nan, inplace=True)
df['refusal_rate_row'].replace([np.inf, -np.inf], np.nan, inplace=True)

# Handle missing values in the DataFrame
df["service_performance"].fillna(df["service_performance"].mean(), inplace=True)
df['health_score'].fillna(df['health_score'].mean(), inplace=True)
df['admission_rate_row'].fillna(0, inplace=True)
df['refusal_rate_row'].fillna(0, inplace=True)

0       0.611650
1       0.578947
2       0.047619
3       0.717105
4       0.362069
          ...   
9995    0.593445
9996    0.895085
9997    0.217769
9998    0.828518
9999    0.040193
Name: refusal_rate_row, Length: 10000, dtype: float64

In [98]:
if "missing value" in df.isnull().any().to_string():
    print(df.isnull().sum().sum())
    print("Missing values is still present in the DataFrame. Please check the data and handle missing values accordingly.")
else:
    print(f"✅ Data is clean and ready for analysis. No missing values: {df.isnull().sum().sum()}")

✅ Data is clean and ready for analysis. No missing values: 399


## Extract JSON concept

In [99]:
# Copy the DataFrame to avoid modifying the original data
data = df.copy()
data.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn,capacity_pressure,admission_rate_row,refusal_rate_row,staff_attendance_rate,risk_score,risk_level,service_performance,staff_absent,health_score
0,PAT-f9c8afa6,Victor Baker,50,2025-07-30,2025-08-13,general_medicine,91,7,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,14,1,2.512195,0.388350,0.611650,0.821429,0.00,Low,27.737864,5.0,60.595562
1,PAT-5290be70,Jeffrey Chandler,29,2025-11-01,2025-11-14,emergency,88,11,1,32,76.0,32.0,44.0,67.0,70.0,39,35.0,4.0,13,1,2.303030,0.421053,0.578947,0.897436,0.25,Low,27.631579,4.0,61.569771
2,PAT-003ce690,Larry Dixon,29,2025-01-19,2025-01-21,ICU,60,1,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,2,1,1.000000,0.952381,0.047619,0.000000,0.75,High,78.047619,34.0,43.047619
3,PAT-18f78014,Kenneth Scott,77,2025-06-24,2025-06-26,general_medicine,61,6,4,43,152.0,43.0,109.0,67.0,66.0,28,22.0,6.0,2,1,3.454545,0.282895,0.717105,0.785714,0.50,Medium,18.236842,6.0,45.772180
4,PAT-69dc0dc1,April Frost,37,2025-11-15,2025-11-19,general_medicine,60,11,3,37,58.0,37.0,21.0,95.0,63.0,28,0.0,28.0,4,1,1.526316,0.637931,0.362069,0.000000,0.50,Medium,60.241379,28.0,36.758621


## Load models from MinIO

In [100]:
def load_bundle_from_minio(minio_key: str):
    """Download .joblib artifacts from MinIO and load them into a dictionary."""
    key = minio_key.strip("/")
    if key.endswith(".joblib"):
        key = key[:-7]  # Remove the .joblib extension
    object_path = f"models/{key}.joblib" if not key.startswith("models/") else f"{key}.joblib"

    response = None
    try:
        response = client.get_object(bucket_name, object_path)
        raw_data = response.read()
        
        # Wrap the raw bytes in io.BytesIO so joblib can read it like a file
        return joblib.load(io.BytesIO(raw_data))
        
    except S3Error as e:
        raise RuntimeError(f"Failed to load model from MinIO: {e}")
    finally:
        if response is not None:
            response.close()
            response.release_conn()

# Usage
bundle_names = [
    "integrity_audit_bundle",
    "patient_churn_bundle",
    "risk_level_bundle",
    "risk_score_bundle",
    "service_performance_bundle",
    "staff_staff_attendance_rate_bundle",
    "staff_staff_morale_bundle"
]

models = {name: load_bundle_from_minio(name) for name in bundle_names}
print(f"✅ Successfully loaded {list(models.keys())} model bundles from MinIO.")

✅ Successfully loaded ['integrity_audit_bundle', 'patient_churn_bundle', 'risk_level_bundle', 'risk_score_bundle', 'service_performance_bundle', 'staff_staff_attendance_rate_bundle', 'staff_staff_morale_bundle'] model bundles from MinIO.


## Prediction

In [101]:
predictions = data[["patient_id"]].copy()

"""
Choose preferred models based on the task type:
    return Classifier model concept -> (random_forest, xgboost, decision_tree, gradient_boosting, logistic)
    return Regression model concept -> (random_forest, xgboost, decision_tree, ridge, linear)
"""
selected_model = "random_forest"

for concept, bundle in models.items():
    submodel = bundle["models"][selected_model]
    model = submodel["pipeline"]
    label_encoder = submodel.get("label_encoder")

    # get 'features' and 'task'
    features = bundle.get("features")
    task = bundle.get("task", "classification")

    missing_features = [f for f in features if f not in data.columns]
    if missing_features:
        raise ValueError(f"Missing required features for model '{concept}': {missing_features}")

    # Split into X and predictions
    X = data[features].replace([np.inf, -np.inf], np.nan).fillna(0)  # Handle inf and NaN values
    raw_predictions = model.predict(X)

    if label_encoder is not None:
        try:
            raw_predictions = label_encoder.inverse_transform(raw_predictions)
        except Exception:
            pass

    predictions[f"{concept}_predicted"] = raw_predictions

    # Define logic classification or return to regression based on task
    if task == "classification" and hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)

        if probabilities.ndim == 2 and probabilities.shape[1] == 2:
            predictions[f"{concept}_probability"] = probabilities[:, 1]
        else:
            predictions[f"{concept}_probability"] = np.asarray(probabilities).max(axis=1)

    print(f"✅ Successfully processed {concept} using {selected_model} model.")

✅ Successfully processed integrity_audit_bundle using random_forest model.
✅ Successfully processed patient_churn_bundle using random_forest model.
✅ Successfully processed risk_level_bundle using random_forest model.
✅ Successfully processed risk_score_bundle using random_forest model.
✅ Successfully processed service_performance_bundle using random_forest model.
✅ Successfully processed staff_staff_attendance_rate_bundle using random_forest model.
✅ Successfully processed staff_staff_morale_bundle using random_forest model.


In [102]:
predictions.head()

,patient_id,integrity_audit_bundle_predicted,integrity_audit_bundle_probability,patient_churn_bundle_predicted,patient_churn_bundle_probability,risk_level_bundle_predicted,risk_level_bundle_probability,risk_score_bundle_predicted,service_performance_bundle_predicted,staff_staff_attendance_rate_bundle_predicted,staff_staff_morale_bundle_predicted
0,PAT-f9c8afa6,0,0.0,1,1.000000,low,1.000000,0.00,1.5,0.821437,51.999541
1,PAT-5290be70,0,0.0,1,1.000000,low,0.997577,0.25,1.5,0.897437,69.999957
2,PAT-003ce690,0,0.0,1,0.829383,high,0.987945,0.75,1.5,0.000000,89.000010
3,PAT-18f78014,0,0.0,1,1.000000,medium,1.000000,0.50,1.5,0.785716,66.000065
4,PAT-69dc0dc1,0,0.0,1,0.772524,medium,0.990527,0.50,1.5,0.000000,63.000465


### Addition business column

In [103]:
# Business summary

summary = pd.DataFrame(
    {
        "metric": [
            "Predicted churn rate",
            "High-risk patients",
            "Average risk score",
            "Average service performance",
            "Average staff morale",
            "Average staff attendance",
            "Integrity audit flags",
        ],
        "value": [
            predictions["patient_churn_bundle_predicted"].mean(),
            predictions["risk_level_bundle_predicted"].astype(str).str.lower().eq("high").sum(),
            predictions["risk_score_bundle_predicted"].mean(),
            predictions["service_performance_bundle_predicted"].mean(),
            predictions["staff_staff_morale_bundle_predicted"].mean(),
            predictions["staff_staff_attendance_rate_bundle_predicted"].mean(),
            predictions["integrity_audit_bundle_predicted"].astype(int).sum()
        ],
    }
)

summary

,metric,value
0,Predicted churn rate,0.601100
1,High-risk patients,2562.000000
2,Average risk score,0.491013
3,Average service performance,1.480050
4,Average staff morale,73.846489
5,Average staff attendance,0.680211
6,Integrity audit flags,3482.000000
